# SMAE Engine Verification

This notebook demonstrates the resilient extraction of nutritional data from the SMAE PDF using the `SMAEEngine` implementation.

In [ ]:
import sys
import os
from datetime import datetime

# Add project root to path for imports
sys.path.append("../../..")

from backend.smae_engine.source_code.main import SMAEEngine
from backend.smae_engine.source_code.schemas import ExtractionRequest
from google.cloud import bigquery
import pandas as pd

## 1. Initialize Engine and Request
The engine uses Pydantic-first validation and implements a page-by-page resilient extraction pipeline.

In [ ]:
engine = SMAEEngine()
gcs_uri = "gs://nutritional-data-sources/SMAE.pdf"

print(f"Engine initialized for: {gcs_uri}")

# Define the request for a specific page range (e.g., pages 5-7)
request = ExtractionRequest(
    gcs_uri=gcs_uri,
    page_range=(5, 7)
)

## 2. Execute Extraction
We will process the requested pages to verify the logic and SCD Type 2 insertion.

In [ ]:
print(f"Executing extraction for pages {request.page_range}...")
response = engine.run(request)

print(f"\n--- Extraction Summary ---")
print(f"Total Items Extracted: {response.metadata.total_items}")
print(f"Failed Pages (DLQ): {len(response.dead_letter_items)}")
print(f"Processing Time: {response.metadata.processing_time_seconds:.2f} seconds.")

## 3. Verify Results in BigQuery
We query the `food_equivalents` table to see the structured data.

In [ ]:
client = bigquery.Client()
query = """
SELECT 
    family_group, 
    food, 
    suggested_quantity, 
    unit, 
    energy_kcal, 
    is_active,
    ingestion_at,
    page_number
FROM `nutritional-partner.nutrimental_information.food_equivalents` 
ORDER BY ingestion_at DESC
LIMIT 20
"""
df = client.query(query).to_dataframe()
df

## 4. Verify Dead Letter Table
Check if any pages failed and were logged for manual review.

In [ ]:
dlq_query = "SELECT * FROM `nutritional-partner.nutrimental_information.extraction_dead_letter` ORDER BY ingestion_at DESC"
dlq_df = client.query(dlq_query).to_dataframe()
if dlq_df.empty:
    print("✅ No extraction errors found in DLQ.")
else:
    print(f"⚠️ Found {len(dlq_df)} errors in DLQ.")
    display(dlq_df)